<a href="https://colab.research.google.com/github/eric20041027/Renting-recommendation-ONNX/blob/main/notebooks/ce_expansion_augment_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CE 語意擴展融入訓練實驗(做法 B + 房源文字富化)

**目標**:驗證「把 extension map 的語意融入 CE 訓練資料」能否提升語意理解(NDCG@5),並讓「想安靜→隔音房源」「可開伙→瓦斯爐房源」這類跨詞對應從推論層拼接轉為模型內化。

**實驗設計(A/B/C 三組,同 test split 比 NDCG@5)**:
| 組 | 訓練資料 | property 文字 | MAX_LENGTH | 說明 |
|---|---|---|---|---|
| **A baseline** | recommendation_train.json | property_to_text(短) | 64 | 重現現有 production 配方 |
| **B query增強** | + 擴展口語 query | property_to_text(短) | 64 | 只增強 query,房源文字不變 |
| **C 富化+增強** | enriched + 擴展口語 | 富化(含 notes) | 128 | 房源特徵全可見,安靜/門禁類能學 |

**前提**:Colab GPU(建議 A100/T4 皆可)。完整跑完 = 3× (訓 teacher + 訓 student + 匯出 + 評估)。

**回滾**:本 notebook 全在 Colab 環境跑,不碰本機 production 模型。實驗結束把 ONNX 下載回本機才會影響 production。

## 0. 環境 — clone repo + 裝依賴 + 確認 GPU

In [ ]:
# ⚠️ 改成你的 repo URL(若為 private,用 token 或先上傳)
REPO_URL = 'https://github.com/eric20041027/Renting-recommendation-ONNX.git'
BRANCH   = 'main'   # 實驗腳本 merge 進 main 後用 main;否則填實驗分支

import os
if not os.path.isdir('Renting-recommendation-ONNX'):
    !git clone -b $BRANCH $REPO_URL
%cd Renting-recommendation-ONNX
!git log --oneline -3

In [ ]:
!pip -q install 'torch>=2.0' 'transformers>=4.35' 'onnx>=1.14' 'onnxruntime>=1.17' 'pandas>=2.0' 'pydantic>=2.0,<3.0' tqdm
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. 生成資料集
- baseline/B 用既有 `recommendation_*.json`(repo 沒附 → 用 generate_dataset 重生一份標準版)。
- C 用富化版 `*_enriched.json` + 富化增強 train。

In [ ]:
# 標準資料集(A baseline / B 用)。data/processed/*.json 被 gitignore,Colab 需重生。
!python pipeline/data_prep/generate_dataset.py

# B:query 增強(短文字版)→ recommendation_train_augmented.json
!python pipeline/data_prep/augment_with_expansion_map.py --write

# C:富化重生 + 富化 query 增強 → *_enriched.json + recommendation_train_enriched_augmented.json
!python pipeline/data_prep/augment_with_expansion_map.py --rebuild-enriched --write --enriched

import json, glob
for f in sorted(glob.glob('data/processed/recommendation_*.json')):
    print(f.split('/')[-1], '=', len(json.load(open(f))))

## 2. 共用:訓練 + 評估 helper

訓練腳本 (`train_and_export_onnx.py`) 的 MAX_LENGTH 與資料路徑、teacher 腳本的資料路徑都寫死。我們用 `sed` 在跑前把它們改成該組設定,跑完還原。每組產出獨立 ONNX。

**A100 純加速(已內建於 `run_group`)**:每組訓練前自動套用 `apply_a100_accel()` — fp16→bf16(A100 原生加速、數值更穩)、TF32(免費 matmul 加速)、eval batch→256、dataloader workers。**train batch 維持 32、epochs/LR 不動** → 不改訓練動態,A/B 結果乾淨可信。

> 為何不加大 train batch:會改變有效學習率與 ListNet/KD 的 batch 統計 → 汙染「擴展增強有沒有用」的判斷,且有重訓回歸前例。任務本身小(rbt3、~28k樣本),純加速 + A100 已足夠快。

In [ ]:
import subprocess, shutil, os, re, json

TRAIN_PY   = 'pipeline/model_training/train_and_export_onnx.py'
TEACHER_PY = 'pipeline/model_training/train_teacher.py'
PROC       = 'data/processed'

def _backup(paths):
    for p in paths: shutil.copy(p, p + '.orig')
def _restore(paths):
    for p in paths:
        if os.path.exists(p + '.orig'): shutil.move(p + '.orig', p)

def set_max_length(n):
    for py in (TRAIN_PY, TEACHER_PY):
        s = open(py).read()
        s = re.sub(r'MAX_LENGTH\s*=\s*\d+', f'MAX_LENGTH = {n}', s, count=1)
        # train_teacher 可能用 max_length=64 字面;一併處理常見寫法
        s = re.sub(r'max_length\s*=\s*64\b', f'max_length={n}', s)
        open(py, 'w').write(s)

def point_data(train_name, dev_name, test_name):
    """把訓練/teacher 腳本讀的 train/dev/test 檔名改成指定組別的檔。"""
    for py in (TRAIN_PY, TEACHER_PY):
        s = open(py).read()
        s = s.replace('recommendation_train.json', train_name)
        s = s.replace('recommendation_dev.json',   dev_name)
        s = s.replace('recommendation_test.json',  test_name)
        open(py, 'w').write(s)

def apply_a100_accel():
    """A100 純加速 — 零風險,不改訓練動態(batch/epochs/LR 都不動),A/B 結果不受汙染。
    只做:fp16→bf16(A100 原生、數值更穩)、加大 eval batch(只前向)、dataloader workers、
    並在腳本頂端啟用 TF32(免費 matmul 加速)。train batch 維持 32 與 production 一致。"""
    for py in (TRAIN_PY, TEACHER_PY):
        s = open(py).read()
        # 1) fp16 → bf16(A100 對 bf16 有原生加速且不易 overflow)
        s = s.replace('fp16=torch.cuda.is_available()', 'bf16=torch.cuda.is_available()')
        # 2) 加大 eval batch(評估只前向,A100 記憶體充裕)+ dataloader workers
        s = re.sub(r'per_device_eval_batch_size\s*=\s*\w+',
                   'per_device_eval_batch_size=256', s)
        if 'dataloader_num_workers' not in s:
            s = s.replace('per_device_eval_batch_size=256',
                          'per_device_eval_batch_size=256,\n        dataloader_num_workers=2,\n        dataloader_pin_memory=True', 1)
        # 3) TF32(免費 matmul/cudnn 加速)— 注入在 import torch 之後
        if 'allow_tf32' not in s:
            s = s.replace('import torch\n',
                          'import torch\ntorch.backends.cuda.matmul.allow_tf32 = True\n'
                          'torch.backends.cudnn.allow_tf32 = True\n', 1)
        open(py, 'w').write(s)
    print('  [A100] bf16 + TF32 + eval_batch=256 + dataloader workers 已套用(train batch 不變)')

def run_group(name, train_name, dev_name, test_name, max_len):
    """訓 teacher → 訓 student → 匯出 ONNX。回傳該組 quant onnx 路徑(已改名隔離)。"""
    print(f'\n===== 訓練組 {name}: train={train_name}, MAX_LENGTH={max_len} =====')
    _backup([TRAIN_PY, TEACHER_PY])
    try:
        set_max_length(max_len)
        point_data(train_name, dev_name, test_name)
        apply_a100_accel()
        # 1) teacher(rbt6)
        subprocess.run(['python', '-m', 'pipeline.model_training.train_teacher'], check=True)
        # 2) student(rbt3)+ 匯出 ONNX
        subprocess.run(['python', '-m', 'pipeline.model_training.train_and_export_onnx'], check=True)
    finally:
        _restore([TRAIN_PY, TEACHER_PY])
    # 隔離 ONNX(預設都輸出到 custom_onnx_model_dir/my_custom_model_quant.onnx)
    src = 'frontend/models/custom_onnx_model_dir/my_custom_model_quant.onnx'
    dst_dir = f'experiment_models/{name}'
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy(src, f'{dst_dir}/model_quant.onnx')
    for tf in ('tokenizer.json','tokenizer_config.json','vocab.txt','config.json','special_tokens_map.json'):
        fp = f'frontend/models/custom_onnx_model_dir/{tf}'
        if os.path.exists(fp): shutil.copy(fp, f'{dst_dir}/{tf}')
    print(f'  ✅ {name} 模型存於 {dst_dir}')
    return dst_dir

In [ ]:
import numpy as np, onnxruntime as ort
from transformers import BertTokenizerFast

def ndcg_eval(model_dir, test_path, max_len, k=5):
    """用該組 ONNX + tokenizer + 對應 test split 算 NDCG@5(同 ce query A/B 框架)。"""
    tok  = BertTokenizerFast.from_pretrained(model_dir)
    sess = ort.InferenceSession(f'{model_dir}/model_quant.onnx', providers=['CPUExecutionProvider'])
    def score(q, t):
        enc = tok(q, t, max_length=max_len, padding='max_length', truncation=True, return_tensors='np')
        logits = sess.run(None, {
            'input_ids': enc['input_ids'].astype(np.int64),
            'attention_mask': enc['attention_mask'].astype(np.int64),
            'token_type_ids': enc.get('token_type_ids', np.zeros_like(enc['input_ids'])).astype(np.int64),
        })[0][0]
        m = float(np.max(logits)); e0,e1 = np.exp(logits[0]-m), np.exp(logits[1]-m)
        return float(e1/(e0+e1))
    data = json.load(open(test_path))
    groups = {}
    for s in data:
        rel = max(0, s.get('relevance', s['label']))
        groups.setdefault(s['query'], []).append((s['property'], float(rel)))
    ev = {q:c for q,c in groups.items() if len(c)>=2 and any(r>0 for _,r in c)}
    def dcg(rs): return sum(r/np.log2(i+2) for i,r in enumerate(rs))
    def ndcg(rs):
        idcg = dcg(sorted(rs, reverse=True)[:k])
        return dcg(rs[:k])/idcg if idcg else 0.0
    scores = []
    for q, cands in ev.items():
        ranked = sorted(((score(q,t), r) for t,r in cands), key=lambda x:x[0], reverse=True)
        scores.append(ndcg([r for _,r in ranked]))
    return float(np.mean(scores)), len(scores)

## 3. 跑三組(可分開跑,各約數分鐘 on A100)

> 若只想先看 B vs A,可跳過 C。每組約 = teacher(~rbt6 蒸餾源)+ student 訓練時間。

In [ ]:
# A baseline:標準資料 + 短文字 + MAX_LENGTH=64
dir_A = run_group('A_baseline',
                  'recommendation_train.json', 'recommendation_dev.json', 'recommendation_test.json',
                  max_len=64)

In [ ]:
# B query增強:增強 train + 短文字 + MAX_LENGTH=64(dev/test 用標準短文字版)
dir_B = run_group('B_query_aug',
                  'recommendation_train_augmented.json', 'recommendation_dev.json', 'recommendation_test.json',
                  max_len=64)

In [ ]:
# C 富化+增強:富化增強 train + 富化 dev/test + MAX_LENGTH=128
dir_C = run_group('C_enriched_aug',
                  'recommendation_train_enriched_augmented.json',
                  'recommendation_dev_enriched.json', 'recommendation_test_enriched.json',
                  max_len=128)

## 4. A/B/C 評估 — NDCG@5 比較

**注意公平性**:A/B 用短文字 test、C 用富化 test。三組各自在「自己訓練時的文字格式」上評估(訓練/推論一致)。要嚴格比 C vs A,額外做交叉評估(C 模型在富化 test、A 模型在短 test),因為換了文字格式無法直接同一份 test 比 —— 這點在報告要講清楚。

In [ ]:
results = {}
results['A_baseline']    = ndcg_eval(dir_A, f'{PROC}/recommendation_test.json',          64)
results['B_query_aug']   = ndcg_eval(dir_B, f'{PROC}/recommendation_test.json',          64)
results['C_enriched_aug']= ndcg_eval(dir_C, f'{PROC}/recommendation_test_enriched.json', 128)

print(f"{'組別':<16}{'NDCG@5':>10}{'#queries':>10}")
for name,(nd,n) in results.items():
    print(f'{name:<16}{nd:>10.4f}{n:>10}')

print('\n--- 對照 ---')
print(f"B vs A (query 增強效果): {results['B_query_aug'][0]-results['A_baseline'][0]:+.4f}")
print(f"C vs A (富化+增強效果):  {results['C_enriched_aug'][0]-results['A_baseline'][0]:+.4f}")
print('\n⚠️ C 換了 test 文字格式(富化),與 A/B 非同一份 test,差值僅供參考。')
print('   嚴格結論看「安靜/門禁類 query 的 per-query NDCG」是否在 C 顯著上升(下一格)。')

## 5. 重點驗證:安靜/門禁/採光類 query 在 C 是否真的學會

這才是你專題的核心論點 —— 富化讓 CE 看得到房源 notes 特徵,「想安靜→隔音房源」能學起來。看這些 query 的 per-query NDCG@5:C 應明顯高於 A。

In [ ]:
FOCUS = ['安靜','怕吵','女生安全','晚歸','採光好','外送族','女生獨居','治安']

def per_query_ndcg(model_dir, test_path, max_len, queries, k=5):
    tok  = BertTokenizerFast.from_pretrained(model_dir)
    sess = ort.InferenceSession(f'{model_dir}/model_quant.onnx', providers=['CPUExecutionProvider'])
    def score(q,t):
        enc = tok(q,t,max_length=max_len,padding='max_length',truncation=True,return_tensors='np')
        lo = sess.run(None,{'input_ids':enc['input_ids'].astype(np.int64),
                            'attention_mask':enc['attention_mask'].astype(np.int64),
                            'token_type_ids':enc.get('token_type_ids',np.zeros_like(enc['input_ids'])).astype(np.int64)})[0][0]
        m=float(np.max(lo)); e0,e1=np.exp(lo[0]-m),np.exp(lo[1]-m); return float(e1/(e0+e1))
    data=json.load(open(test_path)); groups={}
    for s in data:
        groups.setdefault(s['query'],[]).append((s['property'],float(max(0,s.get('relevance',s['label'])))))
    def dcg(rs): return sum(r/np.log2(i+2) for i,r in enumerate(rs))
    out={}
    for q in queries:
        if q not in groups or len(groups[q])<2: continue
        ranked=sorted(((score(q,t),r) for t,r in groups[q]),key=lambda x:x[0],reverse=True)
        rs=[r for _,r in ranked]; idcg=dcg(sorted(rs,reverse=True)[:k])
        out[q]=dcg(rs[:k])/idcg if idcg else 0.0
    return out

a = per_query_ndcg(dir_A, f'{PROC}/recommendation_test.json',          64,  FOCUS)
c = per_query_ndcg(dir_C, f'{PROC}/recommendation_test_enriched.json', 128, FOCUS)
print(f"{'query':<12}{'A baseline':>12}{'C 富化':>10}{'Δ':>9}")
for q in FOCUS:
    if q in a or q in c:
        av,cv=a.get(q,float('nan')),c.get(q,float('nan'))
        print(f'{q:<12}{av:>12.3f}{cv:>10.3f}{(cv-av):>+9.3f}')

## 6. 下載勝出模型(僅在確認提升後)

若某組明顯優於 baseline 才下載回本機替換 production。**注意**:C 組(富化+MAX_LENGTH=128)若採用,前端 `inference-worker.js` 的 MAX_LENGTH 要改 128、且 `scorePair` 要餵富化房源文字(buildPropText),否則訓練/推論不一致。

In [ ]:
from google.colab import files
import shutil
WINNER = 'C_enriched_aug'   # ← 改成勝出組
shutil.make_archive(WINNER, 'zip', f'experiment_models/{WINNER}')
files.download(f'{WINNER}.zip')